<a href="https://colab.research.google.com/github/Rimshakalhoro/flyrank-ml-internship-rimsha/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rimshakalhoro/flyrank-ml-internship-rimsha/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### My lane as an ML task

My provisional lane is Refresh / Content Opportunity Scoring.

I frame this primarily as a ranking and scoring problem. The goal is not simply to classify every page as “good” or “bad.” Instead, the goal is to rank pages so that a content or SEO team can review the most important candidates first.

Each page can receive a score based on observable search-performance and content signals. The final output should be a ranked review queue with supporting reasons. A human reviewer can then decide whether the appropriate action is refresh, expansion, protection, pruning, or monitoring.

I may use classification as one component of the scoring system, but the main decision-support output is a ranked list of pages.

In [7]:
print("ML task type: Ranking / Scoring")
print("Unit: One content page")
print("Output: Ranked review queue")


ML task type: Ranking / Scoring
Unit: One content page
Output: Ranked review queue


### Target or proxy

For the starter dataset, I will use declining performance as a provisional proxy signal.

The proxy label is:

`is_declining = trend_direction == "down"`

This label comes from an observed field in the starter dataset, but it is a defined proxy rather than a perfect future outcome. A page being classified as declining does not prove that refreshing it will improve performance.

For the capstone, I may strengthen this definition by using a future-looking outcome, where features are measured before a decision point and the decline or recovery is measured in a later time window.

The purpose of the current proxy is to help identify pages that may deserve earlier human review.

In [8]:
import pandas as pd

url = "https://raw.githubusercontent.com/Rimshakalhoro/flyrank-ml-internship-rimsha/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

# Create the provisional proxy label

df["is_declining"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Declining pages:", df["is_declining"].sum())
print("Not declining:", (df["is_declining"] == 0).sum())

print("\nDeclining rate:")
print(round(df["is_declining"].mean() * 100, 2), "%")


Dataset loaded successfully!
Rows: 30000
Columns: 44
Declining pages: 16262
Not declining: 13738

Declining rate:
54.21 %


### Success metric

My main success metric will be Precision@50.

This metric answers a practical decision question: among the top 50 pages recommended for review, how many are actually positive according to the chosen proxy label?

I chose Precision@50 because a content team has limited review capacity. The most important part of the ranked list is therefore whether the highest-priority recommendations contain useful candidates.

A higher Precision@50 means that the system places more relevant pages near the top of the review queue.

For the starter dataset, I will compare the result against a simple baseline rather than claiming that one model is automatically good without comparison.

In [9]:
# Define the success metric

K = 50

print("Primary success metric: Precision@50")
print("Interpretation:")
print(f"Of the top {K} recommended pages, what proportion are positive?")


Primary success metric: Precision@50
Interpretation:
Of the top 50 recommended pages, what proportion are positive?


### Unit of analysis

The unit of analysis is one content page.

Each row in my modelling dataframe represents one anonymized content item and contains observable measurements about that page, such as search visibility, clicks, sessions, content age, freshness, CTR, average position, and other available signals.

The goal is to score or rank individual pages for possible human review. The output is therefore page-level rather than client-level or day-level.

In [10]:
# ==========================================
# SECTION 4 — UNIT OF ANALYSIS
# ==========================================

# Keep one row per content page
lane_df = df.drop_duplicates(subset=["content_id"]).copy()

print("Rows in modelling dataframe:", len(lane_df))

print("\nUnique content pages:")
print(lane_df["content_id"].nunique())

print("\nOne row represents:")
print("ONE CONTENT PAGE")

lane_df.head()


Rows in modelling dataframe: 30000

Unique content pages:
30000

One row represents:
ONE CONTENT PAGE


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


### Why ML can beat a fixed rule

A single if-statement cannot easily represent all of the different combinations of signals that may make a page worth reviewing.

For example, one page may have high visibility but declining performance, while another may have strong visibility but low CTR. A third page may be old, stale, and losing engagement. These patterns involve multiple signals interacting with each other.

A fixed rule requires manually choosing thresholds and weights. A machine-learning approach may learn more complex relationships between observable features and the chosen proxy outcome.

However, ML is not automatically better. The learned approach must be compared with a simple baseline and evaluated honestly. The final system should remain decision-support: it prioritizes pages for human review rather than automatically deciding that a page must be changed.

In [11]:
# Compare the idea of a simple rule with multiple signals

possible_signals = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "ctr",
    "avg_position",
    "freshness_tier",
    "engagement_rate"
]

print("Number of possible signals:", len(possible_signals))

print("\nPossible signals used for decision-making:")
for signal in possible_signals:
    print("-", signal)

print("\nConclusion:")
print("Multiple interacting signals make this more complex than one fixed if-statement.")


Number of possible signals: 8

Possible signals used for decision-making:
- impressions_90d
- clicks_90d
- sessions_90d
- content_age_days
- ctr
- avg_position
- freshness_tier
- engagement_rate

Conclusion:
Multiple interacting signals make this more complex than one fixed if-statement.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.